# 02 — Feature Engineering
**TFM "La Dieta de un País" · Aguacate Team · Nuclio Digital School**

---
> **SESIÓN 2 — 03/04/2026**  
> Notebook creado en Sesión 2. Objetivo: transformar los 3 datasets crudos en
> un único parquet maestro listo para clustering y modelado supervisado.

---

## Pipeline de este notebook

```
raw/fbs_raw.csv          ──┐
raw/emissions_raw.csv    ──┤──► ETL ──► processed/02_dataset_final.parquet
raw/cpi_raw.csv          ──┘
```

**Pasos:**
1. Cargar FBS → filtrar → mapear 87 ítems a 7 macrocategorías → vectorizar a %
2. Cargar Emissions → calcular CO2eq/cápita usando población de FBS
3. Cargar CPI → anualizar (media 12 meses)
4. Join de los 3 datasets en un único DataFrame
5. Validación final + guardado en Parquet

**Output:** `processed/02_dataset_final.parquet`  
~390 filas · 30 países · 13 años (2010–2022) · 11 columnas

In [1]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — Imports y configuración
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import gc
import sys
from pathlib import Path

# Rutas del proyecto
ROOT = Path('..') 
RAW  = ROOT / 'raw'
PROC = ROOT / 'processed'
PROC.mkdir(exist_ok=True)

# Importar el mapeo completo generado en Sesión 2
sys.path.append(str(Path('.')))
from mapeo_fao_completo import MAPEO_FAO, ITEMS_AGREGADOS, MACRO_COLS

# Período de análisis
AÑO_MIN, AÑO_MAX = 2010, 2022

# 30 países seleccionados (Japón → Viet Nam, decisión Sesión 1)
PAISES_30 = [
    'Spain', 'Italy', 'France', 'Germany',
    'United Kingdom of Great Britain and Northern Ireland',
    'Poland', 'Romania',
    'United States of America', 'Canada', 'Mexico',
    'Brazil', 'Argentina', 'Colombia',
    'Nigeria', 'Ethiopia', 'Kenya', 'Senegal',
    'Egypt', 'Morocco', 'Saudi Arabia',
    'India', 'Bangladesh', 'Pakistan',
    'China, mainland', 'Republic of Korea', 'Indonesia', 'Thailand', 'Viet Nam',
    'Australia', 'Kazakhstan',
]

print(f'ROOT     : {ROOT.resolve()}')
print(f'RAW      : {RAW.resolve()}')
print(f'PROC     : {PROC.resolve()}')
print(f'Países   : {len(PAISES_30)}')
print(f'Período  : {AÑO_MIN}–{AÑO_MAX}')

[mapeo_fao_completo] ✅ 108 ítems mapeados a 7 categorías.
[mapeo_fao_completo] Distribución:
  Cereales              14 ítems
  Tuberculos             6 ítems
  Azucares               6 ítems
  Aceites_Grasas        24 ítems
  Carnes                15 ítems
  Lacteos_Huevos        12 ítems
  Frutas_Verduras       31 ítems
ROOT     : /Users/macbookdedenys/Desktop/N/TFM_Dieta_Pais
RAW      : /Users/macbookdedenys/Desktop/N/TFM_Dieta_Pais/raw
PROC     : /Users/macbookdedenys/Desktop/N/TFM_Dieta_Pais/processed
Países   : 30
Período  : 2010–2022


---
## PASO 1 — Food Balance Sheets (FBS)
**Input:** `raw/fbs_raw.csv` (604 MB)  
**Output:** vector dietario porcentual por país-año + población

In [2]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 1.1 Lectura eficiente del FBS
# Función de lectura con detección automática de encoding
# (fix encoding UTF-8 implementado en Sesión 1)
# ─────────────────────────────────────────────────────────────
def leer_fao(path, usecols=None):
    """Lee un CSV de FAOSTAT probando encodings hasta encontrar uno limpio."""
    dtypes = {
        'Area': 'category', 'Year': 'int16',
        'Item': 'category', 'Element': 'category', 'Value': 'float32'
    }
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            df = pd.read_csv(path, encoding=enc, usecols=usecols,
                             dtype=dtypes, low_memory=False)
            muestra = df['Area'].dropna().iloc[0] if 'Area' in df.columns else ''
            if 'Ã' not in str(muestra):
                print(f'  Encoding OK: {enc} | {len(df):,} filas')
                return df
        except Exception:
            continue
    raise ValueError(f'No se pudo leer {path} con ningún encoding conocido.')

print('Cargando FBS...')
fbs_raw = leer_fao(
    RAW / 'fbs_raw.csv',
    usecols=['Area', 'Year', 'Item', 'Element', 'Value']
)
print(f'FBS cargado: {fbs_raw.shape}')

Cargando FBS...


  Encoding OK: utf-8 | 4,820,497 filas
FBS cargado: (4820497, 5)


In [3]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 1.2 Extraer población del FBS
# La población está embebida en el propio FBS como elemento
# 'Total Population - Both sexes' (unidad: 1000 personas).
# No necesitamos dataset adicional.
# ─────────────────────────────────────────────────────────────
pop = (
    fbs_raw[
        (fbs_raw['Element'] == 'Total Population - Both sexes') &
        (fbs_raw['Year'] >= AÑO_MIN) & (fbs_raw['Year'] <= AÑO_MAX) &
        (fbs_raw['Area'].isin(PAISES_30))
    ][['Area', 'Year', 'Value']]
    .copy()
    .rename(columns={'Value': 'Population_1000'})
)

print(f'Población extraída: {pop.shape} | Nulos: {pop["Population_1000"].isna().sum()}')
pop.head()

Población extraída: (390, 3) | Nulos: 0


,Area,Year,Population_1000
18,Argentina,2010,41288.691406
26,Argentina,2011,41730.660156
30,Australia,2010,22141.580078
31,Argentina,2012,42161.718750
40,Australia,2011,22479.779297


In [4]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 1.3 Filtrar kcal y excluir ítems agregados
# Fix doble conteo implementado en Sesión 1: excluir 33 agregados.
# ─────────────────────────────────────────────────────────────
fbs_kcal = fbs_raw[
    (fbs_raw['Element'] == 'Food supply (kcal/capita/day)') &
    (~fbs_raw['Item'].isin(ITEMS_AGREGADOS)) &
    (fbs_raw['Year'] >= AÑO_MIN) & (fbs_raw['Year'] <= AÑO_MAX) &
    (fbs_raw['Area'].isin(PAISES_30))
].copy()

print(f'FBS filtrado (kcal, hoja, período, países): {fbs_kcal.shape}')
print(f'Países únicos: {fbs_kcal["Area"].nunique()}')
print(f'Ítems únicos:  {fbs_kcal["Item"].nunique()}')

# Liberar memoria del raw completo
del fbs_raw
gc.collect()

FBS filtrado (kcal, hoja, período, países): (32020, 5)
Países únicos: 30
Ítems únicos:  89


0

In [5]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 1.4 Aplicar mapeo y VALIDAR fugas de categorías
# CRÍTICO: antes de descartar NaN, verificar que los ítems
# no mapeados no superen 50 kcal/día (umbral de ruido).
# ─────────────────────────────────────────────────────────────
fbs_kcal['Macro_Grupo'] = fbs_kcal['Item'].map(MAPEO_FAO)

# Diagnóstico de fugas
no_mapeados = fbs_kcal[fbs_kcal['Macro_Grupo'].isna()]
if len(no_mapeados) > 0:
    print('⚠️  ÍTEMS NO MAPEADOS (verificar si superan 50 kcal/día):')
    fuga = (
        no_mapeados.groupby('Item', observed=True)['Value']
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    fuga.columns = ['Item', 'kcal_media_dia']
    criticos = fuga[fuga['kcal_media_dia'] > 50]
    if len(criticos) > 0:
        print('🔴 CRÍTICO — Ítems con > 50 kcal/día sin mapear:')
        print(criticos.to_string(index=False))
        print('\n→ Añadir estos ítems a MAPEO_FAO en mapeo_fao_completo.py')
    else:
        print('✅ Todos los ítems no mapeados están bajo el umbral de 50 kcal/día.')
        print(fuga.head(10).to_string(index=False))
else:
    print('✅ Todos los ítems tienen categoría asignada.')

# Descartar no mapeados
fbs_kcal = fbs_kcal.dropna(subset=['Macro_Grupo'])
print(f'\nFBS tras descartar no mapeados: {fbs_kcal.shape}')

⚠️  ÍTEMS NO MAPEADOS (verificar si superan 50 kcal/día):
✅ Todos los ítems no mapeados están bajo el umbral de 50 kcal/día.
                      Item  kcal_media_dia
        Fats, Animals, Raw       41.088383
Pulses, Other and products       27.872025
                Groundnuts       23.271999
      Beverages, Alcoholic       21.391342
            Aquatic Plants       20.340769
                    Offals       10.411257
            Maize Germ Oil        9.165699
               Sugar Crops        7.372649
            Palmkernel Oil        5.215087
      Beverages, Fermented        4.130785

FBS tras descartar no mapeados: (27060, 6)


In [6]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 1.5 Agrupar, pivotar y normalizar a porcentajes
# Resultado: cada fila = un país-año con 7 columnas de % dieta
# ─────────────────────────────────────────────────────────────

# Sumar kcal por país, año y macrocategoría
df_grouped = (
    fbs_kcal
    .groupby(['Area', 'Year', 'Macro_Grupo'], observed=True)['Value']
    .sum()
    .reset_index()
)

# Pivotar: cada macrocategoría se convierte en columna
df_vector = (
    df_grouped
    .pivot(index=['Area', 'Year'], columns='Macro_Grupo', values='Value')
    .fillna(0)
    .astype('float32')
    .reset_index()
)

# Asegurar que todas las columnas de macrocategorías existen
for col in MACRO_COLS:
    if col not in df_vector.columns:
        df_vector[col] = 0.0

# Total DES (Dietary Energy Supply) — suma de todas las categorías
df_vector['Total_DES_Kcal'] = df_vector[MACRO_COLS].sum(axis=1).astype('float32')

# Normalizar a porcentajes
for col in MACRO_COLS:
    df_vector[f'pct_{col}'] = (
        (df_vector[col] / df_vector['Total_DES_Kcal'])
        .replace([np.inf, -np.inf], 0)
        .fillna(0)
        .astype('float32')
    )

# Calcular ratio proteína animal/vegetal
df_vector['Ratio_Proteina_AV'] = (
    (df_vector['Carnes'] + df_vector['Lacteos_Huevos']) /
    (df_vector['Frutas_Verduras'] + df_vector['Cereales'] + df_vector['Tuberculos'])
).replace([np.inf, -np.inf], 0).fillna(0).astype('float32')

# Columnas finales del vector dietario
pct_cols = [f'pct_{c}' for c in MACRO_COLS]
df_dieta = df_vector[['Area', 'Year'] + pct_cols + ['Total_DES_Kcal', 'Ratio_Proteina_AV']].copy()

print(f'Vector dietario: {df_dieta.shape}')
print(f'Países: {df_dieta["Area"].nunique()} | Años: {df_dieta["Year"].nunique()}')

# Validación: los porcentajes deben sumar ~1.0
suma_pct = df_dieta[pct_cols].sum(axis=1)
print(f'\nValidación suma %: min={suma_pct.min():.3f} | max={suma_pct.max():.3f} | media={suma_pct.mean():.3f}')
print('(Debe estar próximo a 1.0 en todos los casos)')

del fbs_kcal, df_grouped, df_vector
gc.collect()

Vector dietario: (390, 11)
Países: 30 | Años: 13

Validación suma %: min=1.000 | max=1.000 | media=1.000
(Debe estar próximo a 1.0 en todos los casos)


0

---
## PASO 2 — Emissions (CO2eq/cápita)
**Input:** `raw/emissions_raw.csv` (347 MB)  
**Output:** CO2eq en toneladas per cápita por país-año

In [7]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 2.1 Cargar Emissions y calcular CO2eq/cápita
# Fix proyecciones implementado en Sesión 1: Year <= 2022.
# Fórmula: (CO2eq_kt × 1000) / (Population_1000 × 1000)
# ─────────────────────────────────────────────────────────────
print('Cargando Emissions...')
em_raw = leer_fao(
    RAW / 'emissions_raw.csv',
    usecols=['Area', 'Year', 'Item', 'Element', 'Value']
)

em_co2 = em_raw[
    (em_raw['Element'] == 'Emissions (CO2eq) (AR5)') &
    (em_raw['Item'] == 'Agrifood systems') &
    (em_raw['Year'] >= AÑO_MIN) & (em_raw['Year'] <= AÑO_MAX) &
    (em_raw['Area'].isin(PAISES_30))
][['Area', 'Year', 'Value']].copy()

em_co2.columns = ['Area', 'Year', 'CO2eq_kt']

# Unir con población para calcular per cápita
em_pop = em_co2.merge(pop, on=['Area', 'Year'], how='inner')

# Conversión: kt → t, 1000 personas → personas
em_pop['CO2eq_t_per_capita'] = (
    (em_pop['CO2eq_kt'] * 1000) /
    (em_pop['Population_1000'] * 1000)
).astype('float32')

df_emisiones = em_pop[['Area', 'Year', 'CO2eq_t_per_capita']].copy()

print(f'Emisiones: {df_emisiones.shape} | Nulos: {df_emisiones["CO2eq_t_per_capita"].isna().sum()}')
print(f'Rango CO2eq/cápita: {df_emisiones["CO2eq_t_per_capita"].min():.2f} – {df_emisiones["CO2eq_t_per_capita"].max():.2f} t/persona/año')

del em_raw, em_co2, em_pop
gc.collect()

Cargando Emissions...


  Encoding OK: utf-8 | 2,500,090 filas
Emisiones: (390, 3) | Nulos: 0
Rango CO2eq/cápita: 0.78 – 12.89 t/persona/año


0

---
## PASO 3 — Consumer Price Index (CPI)
**Input:** `raw/cpi_raw.csv` (35.5 MB)  
**Output:** Food CPI anual por país-año (media de 12 meses)

In [8]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 3.1 Cargar CPI y anualizar
# El CPI es mensual; el FBS es anual.
# Calculamos la media de los 12 meses por país-año antes del join.
# ─────────────────────────────────────────────────────────────
print('Cargando CPI...')
cpi_raw = leer_fao(
    RAW / 'cpi_raw.csv',
    usecols=['Area', 'Year', 'Item', 'Element', 'Value']
)

CPI_ITEM = 'Consumer Prices, Food Indices (2015 = 100)'

cpi_food = cpi_raw[
    (cpi_raw['Item'] == CPI_ITEM) &
    (cpi_raw['Year'] >= AÑO_MIN) & (cpi_raw['Year'] <= AÑO_MAX) &
    (cpi_raw['Area'].isin(PAISES_30))
].copy()

# Anualizar: media de los 12 meses
cpi_anual = (
    cpi_food
    .groupby(['Area', 'Year'], observed=True)['Value']
    .mean()
    .reset_index()
    .rename(columns={'Value': 'Food_CPI'})
)
cpi_anual['Food_CPI'] = cpi_anual['Food_CPI'].astype('float32')

print(f'CPI anual: {cpi_anual.shape}')
print(f'Países con CPI: {cpi_anual["Area"].nunique()} de {len(PAISES_30)}')

# Diagnóstico de huecos
paises_sin_cpi = set(PAISES_30) - set(cpi_anual['Area'].unique())
if paises_sin_cpi:
    print(f'⚠️  Países sin CPI: {paises_sin_cpi}')
    print('   → Se imputará con la media regional en el join final.')
else:
    print('✅ Todos los países tienen datos de CPI.')

del cpi_raw, cpi_food
gc.collect()

Cargando CPI...
  Encoding OK: utf-8 | 237,814 filas
CPI anual: (390, 3)
Países con CPI: 30 de 30
✅ Todos los países tienen datos de CPI.


0

---
## PASO 4 — Join maestro de los 3 datasets
Unimos FBS (dieta) + Emissions (CO2eq) + CPI en un único DataFrame.

In [9]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 4.1 Join FBS + Emissions
# Left join desde el vector dietario: base de verdad.
# ─────────────────────────────────────────────────────────────
df_master = df_dieta.merge(df_emisiones, on=['Area', 'Year'], how='left')

nulos_co2 = df_master['CO2eq_t_per_capita'].isna().sum()
print(f'Tras join FBS+Emissions: {df_master.shape} | Nulos CO2eq: {nulos_co2}')
if nulos_co2 > 0:
    print('Filas sin CO2eq (país-año):')
    print(df_master[df_master['CO2eq_t_per_capita'].isna()][['Area','Year']].to_string())

Tras join FBS+Emissions: (390, 12) | Nulos CO2eq: 0


In [10]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 4.2 Join con CPI + imputación por media regional
# Si un país no tiene CPI, se imputa con la media del año
# (estrategia conservadora; documentar en la memoria).
# ─────────────────────────────────────────────────────────────
df_master = df_master.merge(cpi_anual, on=['Area', 'Year'], how='left')

nulos_cpi = df_master['Food_CPI'].isna().sum()
print(f'Tras join con CPI: {df_master.shape} | Nulos CPI: {nulos_cpi}')

if nulos_cpi > 0:
    # Imputar con media del año (proxy regional)
    media_anual = df_master.groupby('Year')['Food_CPI'].transform('mean')
    df_master['Food_CPI'] = df_master['Food_CPI'].fillna(media_anual).astype('float32')
    print(f'  → {nulos_cpi} valores CPI imputados con media anual.')
    print('  → Documentar en la memoria: imputación conservadora por media de año.')

print(f'\nDataset maestro final: {df_master.shape}')
print(f'Nulos totales: {df_master.isna().sum().sum()}')

Tras join con CPI: (390, 13) | Nulos CPI: 0

Dataset maestro final: (390, 13)
Nulos totales: 0


---
## PASO 5 — Validación final y guardado

In [11]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 5.1 Validación final del dataset maestro
# ─────────────────────────────────────────────────────────────
print('═' * 55)
print('VALIDACIÓN FINAL — 02_dataset_final')
print('═' * 55)
print(f'Filas          : {len(df_master)}')
print(f'Columnas       : {df_master.shape[1]}')
print(f'Países         : {df_master["Area"].nunique()}')
print(f'Años           : {sorted(df_master["Year"].unique())}')
print(f'Nulos totales  : {df_master.isna().sum().sum()}')
print()
print('Estadísticas clave:')
print(df_master[pct_cols + ['Total_DES_Kcal', 'CO2eq_t_per_capita', 'Food_CPI']].describe().round(3))
print()

# Verificar que pct suman ~1.0
suma_pct_final = df_master[pct_cols].sum(axis=1)
print(f'Suma % dieta: min={suma_pct_final.min():.3f} | max={suma_pct_final.max():.3f}')

# Verificar rango DES razonable
des_min = df_master['Total_DES_Kcal'].min()
des_max = df_master['Total_DES_Kcal'].max()
print(f'DES range    : {des_min:.0f} – {des_max:.0f} kcal/cápita/día')
if des_min < 1500 or des_max > 4500:
    print('⚠️  Rango DES fuera de lo esperado — revisar filtrado de agregados.')
else:
    print('✅ Rango DES dentro de parámetros normales (1500–4500 kcal).')

═══════════════════════════════════════════════════════
VALIDACIÓN FINAL — 02_dataset_final
═══════════════════════════════════════════════════════
Filas          : 390
Columnas       : 13
Países         : 30
Años           : [np.int16(2010), np.int16(2011), np.int16(2012), np.int16(2013), np.int16(2014), np.int16(2015), np.int16(2016), np.int16(2017), np.int16(2018), np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022)]
Nulos totales  : 0

Estadísticas clave:
       pct_Cereales  pct_Tuberculos  pct_Azucares  pct_Aceites_Grasas  \
count       390.000         390.000       390.000             390.000   
mean          0.425           0.044         0.105               0.123   
std           0.153           0.050         0.041               0.043   
min           0.219           0.006         0.025               0.023   
25%           0.288           0.020         0.082               0.093   
50%           0.410           0.031         0.099               0.126   
75%           

In [12]:
# ─────────────────────────────────────────────────────────────
# SESIÓN 2 — 5.2 Guardar en Parquet
# Este parquet es el input directo del clustering (notebook 03)
# y del modelo supervisado (notebook 04).
# ─────────────────────────────────────────────────────────────
output_path = PROC / '02_dataset_final.parquet'
df_master.to_parquet(output_path, index=False)

size_mb = output_path.stat().st_size / (1024**2)
print(f'✅ Guardado: {output_path}')
print(f'   Tamaño  : {size_mb:.2f} MB')
print(f'   Filas   : {len(df_master)}')
print(f'   Columnas: {list(df_master.columns)}')

del df_dieta, df_emisiones, cpi_anual, df_master, pop
gc.collect()

print('\n[SESIÓN 2] Feature Engineering completado.')
print('Siguiente paso: notebooks/03_clustering.ipynb')

✅ Guardado: ../processed/02_dataset_final.parquet
   Tamaño  : 0.03 MB
   Filas   : 390
   Columnas: ['Area', 'Year', 'pct_Cereales', 'pct_Tuberculos', 'pct_Azucares', 'pct_Aceites_Grasas', 'pct_Carnes', 'pct_Lacteos_Huevos', 'pct_Frutas_Verduras', 'Total_DES_Kcal', 'Ratio_Proteina_AV', 'CO2eq_t_per_capita', 'Food_CPI']

[SESIÓN 2] Feature Engineering completado.
Siguiente paso: notebooks/03_clustering.ipynb
